In [1]:
import pandas as pd
from pathlib import Path

In [2]:
# ============================================================
# 1. PATH CONFIG FOR PROMPT 3
# ============================================================

BASE_DIR = Path("..")

PROMPT_ID = 3

PREPROCESSED_PATH = BASE_DIR / "dataset" / "asap" / "training_set_rel3_preprocessed.csv"
PROMPT_SCORE_PATH = BASE_DIR / "dataset" / "asap++" / "Prompt-3.csv"

OUTPUT_DIR = BASE_DIR / "dataset" / "asap++"
OUTPUT_PATH = OUTPUT_DIR / "Prompt-3-concat.csv"

print("Prompt ID:", PROMPT_ID)
print("Preprocessed path:", PREPROCESSED_PATH)
print("Prompt score path:", PROMPT_SCORE_PATH)
print("Output path:", OUTPUT_PATH)

Prompt ID: 3
Preprocessed path: ..\dataset\asap\training_set_rel3_preprocessed.csv
Prompt score path: ..\dataset\asap++\Prompt-3.csv
Output path: ..\dataset\asap++\Prompt-3-concat.csv


In [3]:
# ============================================================
# 2. HELPER FUNCTIONS
# ============================================================

def normalize_columns(df: pd.DataFrame) -> pd.DataFrame:
    """
    Chuẩn hóa tên cột:
    - strip khoảng trắng
    - lowercase
    - thay khoảng trắng bằng _
    - bỏ ký tự đặc biệt
    """
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(r"\s+", "_", regex=True)
        .str.replace(r"[^\w_]", "", regex=True)
    )
    return df


def find_column(df: pd.DataFrame, candidates: list[str]) -> str:
    """
    Tìm cột theo nhiều tên có thể xuất hiện trong file gốc.
    """
    existing_cols = set(df.columns)

    for col in candidates:
        if col in existing_cols:
            return col

    raise ValueError(
        f"Không tìm thấy cột phù hợp.\n"
        f"Candidates: {candidates}\n"
        f"Existing columns: {list(df.columns)}"
    )


def preview_df(df: pd.DataFrame, name: str, n: int = 5):
    print(f"\n===== {name} =====")
    print("Shape:", df.shape)
    print("Columns:", list(df.columns))
    display(df.head(n))

In [4]:
# ============================================================
# 3. LOAD DATA
# ============================================================

pre = pd.read_csv(PREPROCESSED_PATH)
scores = pd.read_csv(PROMPT_SCORE_PATH)

preview_df(pre, "Raw preprocessed")
preview_df(scores, f"Raw Prompt-{PROMPT_ID} scores")


===== Raw preprocessed =====
Shape: (12976, 5)
Columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,1,1,"Dear local newspaper, I think effects computer...",8.0,0.6
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9.0,0.7
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7.0,0.5
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",10.0,0.8
4,5,1,"Dear @LOCATION1, I know having computers has a...",8.0,0.6



===== Raw Prompt-3 scores =====
Shape: (1726, 5)
Columns: ['Essay ID', 'Content', 'Prompt Adherence', 'Language', 'Narrativity']


,Essay ID,Content,Prompt Adherence,Language,Narrativity
0,5978,0,0,1,1
1,5979,3,3,2,2
2,5980,2,2,2,1
3,5981,0,1,0,0
4,5982,1,1,1,1


In [5]:
# ============================================================
# 4. NORMALIZE COLUMN NAMES
# ============================================================

pre = normalize_columns(pre)
scores = normalize_columns(scores)

preview_df(pre, "Normalized preprocessed")
preview_df(scores, f"Normalized Prompt-{PROMPT_ID} scores")


===== Normalized preprocessed =====
Shape: (12976, 5)
Columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,1,1,"Dear local newspaper, I think effects computer...",8.0,0.6
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9.0,0.7
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7.0,0.5
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",10.0,0.8
4,5,1,"Dear @LOCATION1, I know having computers has a...",8.0,0.6



===== Normalized Prompt-3 scores =====
Shape: (1726, 5)
Columns: ['essay_id', 'content', 'prompt_adherence', 'language', 'narrativity']


,essay_id,content,prompt_adherence,language,narrativity
0,5978,0,0,1,1
1,5979,3,3,2,2
2,5980,2,2,2,1
3,5981,0,1,0,0
4,5982,1,1,1,1


In [6]:
# ============================================================
# 5. VALIDATE PREPROCESSED COLUMNS
# ============================================================

required_pre_cols = [
    "essay_id",
    "essay_set",
    "essay",
    "domain1_score",
    "domain1_score_norm"
]

missing_pre_cols = [col for col in required_pre_cols if col not in pre.columns]

if missing_pre_cols:
    raise ValueError(
        f"File preprocessed thiếu cột: {missing_pre_cols}\n"
        f"Các cột hiện có: {list(pre.columns)}"
    )

print("Preprocessed file OK.")

Preprocessed file OK.


In [7]:
# ============================================================
# 6. MAP PROMPT 3 SCORE COLUMNS
# ============================================================

essay_id_col = find_column(
    scores,
    [
        "essay_id",
        "essayid",
        "essay"
    ]
)

content_col = find_column(
    scores,
    [
        "content"
    ]
)

prompt_adherence_col = find_column(
    scores,
    [
        "prompt_adherence",
        "promptadherence",
        "adherence",
        "prompt"
    ]
)

language_col = find_column(
    scores,
    [
        "language",
        "lang"
    ]
)

narrativity_col = find_column(
    scores,
    [
        "narrativity",
        "narrative",
        "narrat"
    ]
)

print("Column mapping:")
print("essay_id:", essay_id_col)
print("content:", content_col)
print("prompt_adherence:", prompt_adherence_col)
print("language:", language_col)
print("narrativity:", narrativity_col)

Column mapping:
essay_id: essay_id
content: content
prompt_adherence: prompt_adherence
language: language
narrativity: narrativity


In [8]:
# ============================================================
# 7. CLEAN PROMPT 3 SCORE FILE
# ============================================================

prefix = f"p{PROMPT_ID}"

scores_clean = scores.rename(columns={
    essay_id_col: "essay_id",
    content_col: f"{prefix}_content",
    prompt_adherence_col: f"{prefix}_prompt_adherence",
    language_col: f"{prefix}_language",
    narrativity_col: f"{prefix}_narrativity"
})

score_cols = [
    f"{prefix}_content",
    f"{prefix}_prompt_adherence",
    f"{prefix}_language",
    f"{prefix}_narrativity"
]

scores_clean = scores_clean[
    ["essay_id"] + score_cols
].copy()

preview_df(scores_clean, f"Clean Prompt-{PROMPT_ID} scores")


===== Clean Prompt-3 scores =====
Shape: (1726, 5)
Columns: ['essay_id', 'p3_content', 'p3_prompt_adherence', 'p3_language', 'p3_narrativity']


,essay_id,p3_content,p3_prompt_adherence,p3_language,p3_narrativity
0,5978,0,0,1,1
1,5979,3,3,2,2
2,5980,2,2,2,1
3,5981,0,1,0,0
4,5982,1,1,1,1


In [9]:
# ============================================================
# 8. TYPE CONVERSION
# ============================================================

pre["essay_id"] = pd.to_numeric(pre["essay_id"], errors="coerce").astype("Int64")
pre["essay_set"] = pd.to_numeric(pre["essay_set"], errors="coerce").astype("Int64")
pre["domain1_score"] = pd.to_numeric(pre["domain1_score"], errors="coerce")
pre["domain1_score_norm"] = pd.to_numeric(pre["domain1_score_norm"], errors="coerce")

scores_clean["essay_id"] = pd.to_numeric(
    scores_clean["essay_id"],
    errors="coerce"
).astype("Int64")

for col in score_cols:
    scores_clean[col] = pd.to_numeric(scores_clean[col], errors="coerce")

preview_df(pre, "Preprocessed after type conversion")
preview_df(scores_clean, f"Prompt-{PROMPT_ID} scores after type conversion")


===== Preprocessed after type conversion =====
Shape: (12976, 5)
Columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm
0,1,1,"Dear local newspaper, I think effects computer...",8.0,0.6
1,2,1,"Dear @CAPS1 @CAPS2, I believe that using compu...",9.0,0.7
2,3,1,"Dear, @CAPS1 @CAPS2 @CAPS3 More and more peopl...",7.0,0.5
3,4,1,"Dear Local Newspaper, @CAPS1 I have found that...",10.0,0.8
4,5,1,"Dear @LOCATION1, I know having computers has a...",8.0,0.6



===== Prompt-3 scores after type conversion =====
Shape: (1726, 5)
Columns: ['essay_id', 'p3_content', 'p3_prompt_adherence', 'p3_language', 'p3_narrativity']


,essay_id,p3_content,p3_prompt_adherence,p3_language,p3_narrativity
0,5978,0,0,1,1
1,5979,3,3,2,2
2,5980,2,2,2,1
3,5981,0,1,0,0
4,5982,1,1,1,1


In [10]:
# ============================================================
# 9. FILTER PREPROCESSED TO PROMPT 3 ONLY
# ============================================================

pre_prompt = pre[pre["essay_set"] == PROMPT_ID].copy()

print(f"Rows in preprocessed Prompt {PROMPT_ID}:", len(pre_prompt))
print("Unique essay_set:", pre_prompt["essay_set"].unique())

preview_df(pre_prompt, f"Preprocessed Prompt {PROMPT_ID}")

Rows in preprocessed Prompt 3: 1726
Unique essay_set: <IntegerArray>
[3]
Length: 1, dtype: Int64

===== Preprocessed Prompt 3 =====
Shape: (1726, 5)
Columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm
3583,5978,3,The features of the setting affect the cyclist...,1.0,0.33
3584,5979,3,The features of the setting affected the cycli...,2.0,0.67
3585,5980,3,Everyone travels to unfamiliar places. Sometim...,1.0,0.33
3586,5981,3,I believe the features of the cyclist affected...,1.0,0.33
3587,5982,3,The setting effects the cyclist because of the...,2.0,0.67


In [11]:
# ============================================================
# 10. CHECK DUPLICATES BEFORE MERGE
# ============================================================

pre_dup_count = pre_prompt["essay_id"].duplicated().sum()
score_dup_count = scores_clean["essay_id"].duplicated().sum()

print("Duplicate essay_id in pre_prompt:", pre_dup_count)
print("Duplicate essay_id in scores_clean:", score_dup_count)

if pre_dup_count > 0:
    duplicated_ids = (
        pre_prompt
        .loc[pre_prompt["essay_id"].duplicated(), "essay_id"]
        .head(10)
        .tolist()
    )
    raise ValueError(
        f"Preprocessed Prompt {PROMPT_ID} có essay_id bị duplicate. "
        f"Ví dụ: {duplicated_ids}"
    )

if score_dup_count > 0:
    duplicated_ids = (
        scores_clean
        .loc[scores_clean["essay_id"].duplicated(), "essay_id"]
        .head(10)
        .tolist()
    )
    raise ValueError(
        f"Prompt-{PROMPT_ID}.csv có essay_id bị duplicate. "
        f"Ví dụ: {duplicated_ids}"
    )

Duplicate essay_id in pre_prompt: 0
Duplicate essay_id in scores_clean: 0


In [12]:
# ============================================================
# 11. MERGE HORIZONTAL BY essay_id
# ============================================================

concat_prompt = pre_prompt.merge(
    scores_clean,
    on="essay_id",
    how="inner"
)

print("Rows after merge:", len(concat_prompt))
preview_df(concat_prompt, f"Merged Prompt {PROMPT_ID}")

Rows after merge: 1726

===== Merged Prompt 3 =====
Shape: (1726, 9)
Columns: ['essay_id', 'essay_set', 'essay', 'domain1_score', 'domain1_score_norm', 'p3_content', 'p3_prompt_adherence', 'p3_language', 'p3_narrativity']


,essay_id,essay_set,essay,domain1_score,domain1_score_norm,p3_content,p3_prompt_adherence,p3_language,p3_narrativity
0,5978,3,The features of the setting affect the cyclist...,1.0,0.33,0,0,1,1
1,5979,3,The features of the setting affected the cycli...,2.0,0.67,3,3,2,2
2,5980,3,Everyone travels to unfamiliar places. Sometim...,1.0,0.33,2,2,2,1
3,5981,3,I believe the features of the cyclist affected...,1.0,0.33,0,1,0,0
4,5982,3,The setting effects the cyclist because of the...,2.0,0.67,1,1,1,1


In [13]:
# ============================================================
# 12. CHECK MISSING IDS
# ============================================================

pre_ids = set(pre_prompt["essay_id"].dropna().astype(int))
score_ids = set(scores_clean["essay_id"].dropna().astype(int))
merged_ids = set(concat_prompt["essay_id"].dropna().astype(int))

missing_in_scores = sorted(list(pre_ids - score_ids))
missing_in_pre = sorted(list(score_ids - pre_ids))

print(f"Prompt {PROMPT_ID} essays in preprocessed:", len(pre_ids))
print(f"Prompt {PROMPT_ID} essays in score file:", len(score_ids))
print("Merged essays:", len(merged_ids))

print("IDs có trong preprocessed nhưng không có trong score file:", len(missing_in_scores))
print("IDs có trong score file nhưng không có trong preprocessed prompt:", len(missing_in_pre))

if len(missing_in_scores) > 0:
    print("Ví dụ missing_in_scores:", missing_in_scores[:20])

if len(missing_in_pre) > 0:
    print("Ví dụ missing_in_pre:", missing_in_pre[:20])

Prompt 3 essays in preprocessed: 1726
Prompt 3 essays in score file: 1726
Merged essays: 1726
IDs có trong preprocessed nhưng không có trong score file: 0
IDs có trong score file nhưng không có trong preprocessed prompt: 0


In [14]:
# ============================================================
# 13. DROP NaN TRAIT SCORES
# ============================================================

before_drop = len(concat_prompt)

concat_prompt = concat_prompt.dropna(subset=score_cols).copy()

after_drop = len(concat_prompt)

print("Rows before drop NaN trait scores:", before_drop)
print("Rows after drop NaN trait scores:", after_drop)
print("Dropped rows:", before_drop - after_drop)

Rows before drop NaN trait scores: 1726
Rows after drop NaN trait scores: 1726
Dropped rows: 0


In [15]:
# ============================================================
# 14. VALIDATE PROMPT 3 TRAIT SCORE RANGE
# ============================================================

# Prompt 3 trait scores theo guideline là 0-3
for col in score_cols:
    invalid_mask = ~concat_prompt[col].between(0, 3)
    invalid_count = invalid_mask.sum()

    print(f"{col} invalid count:", invalid_count)

    if invalid_count > 0:
        display(concat_prompt.loc[invalid_mask, ["essay_id", col]].head(20))

p3_content invalid count: 0
p3_prompt_adherence invalid count: 0
p3_language invalid count: 0
p3_narrativity invalid count: 0


In [16]:
# ============================================================
# 15. ADD prompt_id AND REORDER COLUMNS
# ============================================================

concat_prompt["prompt_id"] = PROMPT_ID

final_cols = [
    "essay_id",
    "essay_set",
    "prompt_id",
    "essay",
    "domain1_score",
    "domain1_score_norm",
] + score_cols

concat_prompt = concat_prompt[final_cols].copy()

preview_df(concat_prompt, f"Final Prompt-{PROMPT_ID} concat")


===== Final Prompt-3 concat =====
Shape: (1726, 10)
Columns: ['essay_id', 'essay_set', 'prompt_id', 'essay', 'domain1_score', 'domain1_score_norm', 'p3_content', 'p3_prompt_adherence', 'p3_language', 'p3_narrativity']


,essay_id,essay_set,prompt_id,essay,domain1_score,domain1_score_norm,p3_content,p3_prompt_adherence,p3_language,p3_narrativity
0,5978,3,3,The features of the setting affect the cyclist...,1.0,0.33,0,0,1,1
1,5979,3,3,The features of the setting affected the cycli...,2.0,0.67,3,3,2,2
2,5980,3,3,Everyone travels to unfamiliar places. Sometim...,1.0,0.33,2,2,2,1
3,5981,3,3,I believe the features of the cyclist affected...,1.0,0.33,0,1,0,0
4,5982,3,3,The setting effects the cyclist because of the...,2.0,0.67,1,1,1,1


In [17]:
# ============================================================
# 16. SAVE OUTPUT
# ============================================================

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

concat_prompt.to_csv(OUTPUT_PATH, index=False)

print("Saved to:", OUTPUT_PATH)
print("Final shape:", concat_prompt.shape)

Saved to: ..\dataset\asap++\Prompt-3-concat.csv
Final shape: (1726, 10)


In [18]:
# ============================================================
# 17. QUICK SANITY CHECK
# ============================================================

print("Essay set counts:")
print(concat_prompt["essay_set"].value_counts(dropna=False))

print("\nPrompt id counts:")
print(concat_prompt["prompt_id"].value_counts(dropna=False))

print("\nTrait score describe:")
display(concat_prompt[score_cols].describe())

print("\nNaN count:")
print(concat_prompt.isna().sum())

Essay set counts:
essay_set
3    1726
Name: count, dtype: Int64

Prompt id counts:
prompt_id
3    1726
Name: count, dtype: int64

Trait score describe:


,p3_content,p3_prompt_adherence,p3_language,p3_narrativity
count,1726.000000,1726.000000,1726.000000,1726.000000
mean,1.434531,1.478563,1.472190,1.440904
std,0.838944,0.876325,0.846533,0.893608
min,0.000000,0.000000,0.000000,0.000000
25%,1.000000,1.000000,1.000000,1.000000
50%,2.000000,2.000000,2.000000,1.000000
75%,2.000000,2.000000,2.000000,2.000000
max,3.000000,3.000000,3.000000,3.000000



NaN count:
essay_id               0
essay_set              0
prompt_id              0
essay                  0
domain1_score          0
domain1_score_norm     0
p3_content             0
p3_prompt_adherence    0
p3_language            0
p3_narrativity         0
dtype: int64
